# Pooling results into an aggregate measurement

In [1]:
import os
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np

In [2]:
REPORTS_PATH = 'reports'

In [3]:
fs = [os.path.join(REPORTS_PATH, f) for f in os.listdir(REPORTS_PATH) if (not f.startswith('.')) and (f.endswith('.csv'))]
len(fs)

8

In [4]:
df = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
df.head()

,corpus,param,beta,std,k,se,t
0,CABNC,Intercept,0.125855,1.389172,43167739,0.000211,595.243509
1,CABNC,nx,0.301516,0.065247,43167739,0.000010,30361.772347
2,CABNC,ny,-0.058792,0.084814,43167739,0.000013,-4554.381943
3,CABNC,self,-0.359632,2.049051,43167739,0.000312,-1153.148297
4,callfriend-eng_n,Intercept,-0.256187,0.187566,5420692,0.000081,-3180.029702


In [5]:
data = []

In [6]:
for param in tqdm(df['param'].unique()):
    sub = df.loc[df['param'].isin([param])]
    var = (sub['std'] ** 2) * (sub['k'] - 1)
    beta = (sub['beta'] * sub['k'])

    var = var.sum() / (sub['k'] - 1).sum()
    beta = beta.sum() / sub['k'].sum()

    data += [
        {
            'param': param,
            'beta': beta,
            'std': np.sqrt(var),
            'se': np.sqrt(var / (sub['k'] - 1).sum())
        }
    ]

data = pd.DataFrame(data)

  0%|          | 0/4 [00:00<?, ?it/s]

In [7]:
data['t'] = data['beta'] / data['se']

In [8]:
data.head(n=10)

,param,beta,std,se,t
0,Intercept,-2.693007e+07,6.116970e+09,372856.809851,-72.226314
1,nx,2.598474e-01,3.915566e-02,0.000002,108872.481415
2,ny,-4.229476e-02,3.563142e-02,0.000002,-19473.666283
3,self,2.693007e+07,6.116970e+09,372856.809851,72.226314
